#  DiffPriv-Gateway: Scalability & Multi-Dataset Audit
**Project Phase:** 2 (Modular Refactoring & Scalability)  
**Objective:** Validation of the Privacy Engine across diverse business sectors (Finance, HR, IoT).

---

## 1. Executive Summary & SME Context
Σε αυτό το notebook, μεταφέρουμε τον μηχανισμό **Differential Privacy** από το στάδιο της δοκιμής στην πλήρη παραγωγική κλίμακα. Στόχος μας είναι να αποδείξουμε ότι οι Μικρομεσαίες Επιχειρήσεις (SMEs) μπορούν να προστατεύσουν ευαίσθητα δεδομένα από διαφορετικά τμήματα χρησιμοποιώντας μια ενιαία, modular υποδομή.



## 2. Scalability Strategy (The Modular Approach)
Σε αντίθεση με την αρχική προσέγγιση στο `fire.csv`, εδώ υλοποιούμε μια αυτοματοποιημένη ροή (**Pipeline**) που εφαρμόζει τον μηχανισμό Laplace στα εξής σενάρια:

* **Human Resources (`salary.csv`):** Προστασία των μισθολογικών δεδομένων για την αποφυγή διαρροής εισοδήματος εργαζομένων.
* **Banking & Finance (`loans.csv`):** Ανωνυμοποίηση των ποσών δανείων για ασφαλή πιστωτική αξιολόγηση.
* **Demographics (`adult.csv`):** Προστασία εκπαιδευτικών και κοινωνικών χαρακτηριστικών για στατιστική έρευνα.

## 3. Technical Parameters for SME Deployment
| Dataset | Target Sensitive Column | Sensitivity ($\Delta f$) | Industry Sector |
| :--- | :--- | :--- | :--- |
| **fire.csv** | Final Priority | 3 | Public Safety / IoT |
| **salary.csv** | Income | *TBD* | Human Resources |
| **loans.csv** | Loan Amount | *TBD* | Finance |
| **adult.csv** | Education-Num | *TBD* | Market Research |

---

In [2]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. THE PRIVACY ENGINE (Η Core Συνάρτησή μας)
# ==========================================
def apply_laplace_mechanism(df, column, epsilon, sensitivity, min_val=None, max_val=None):
    """Εφαρμόζει θόρυβο Laplace με δυνατότητα Clipping."""
    beta = sensitivity / epsilon
    noise = np.random.laplace(0, beta, size=len(df))
    
    new_col_name = f'DP_{column}'
    df[new_col_name] = df[column] + noise
    
    # Εφαρμογή Clipping αν έχουν οριστεί όρια
    if min_val is not None and max_val is not None:
        df[new_col_name] = np.clip(df[new_col_name], a_min=min_val, a_max=max_val)
        df[new_col_name] = np.round(df[new_col_name]) # Στρογγυλοποίηση για ακέραιες κατηγορίες
        
    return df

# ==========================================
# 2. ΤΟ ΚΕΝΤΡΟ ΕΛΕΓΧΟΥ (Configuration)
# Εδώ η Αθηνά θα συμπληρώνει τα 'None' μόλις κάνει την ανάλυσή της
# ==========================================
datasets_config = {
    'fire': {
        'path': '../data/synthetic/fire.csv',
        'target_col': 'Final Priority',
        'epsilon': 1.0,
        'sensitivity': 3.0,
        'min_val': 0,
        'max_val': 3
    },
    'salary': {
        'path': '../data/synthetic/salary.csv',
        'target_col': 'Income', 
        'epsilon': 1.0,
        'sensitivity': None,  # TBD από την Αθηνά
        'min_val': None,      # TBD
        'max_val': None       # TBD
    },
    'loans': {
        'path': '../data/synthetic/loans.csv',
        'target_col': 'LoanAmount', 
        'epsilon': 1.2,
        'sensitivity': None,  # TBD
        'min_val': None,      # TBD
        'max_val': None       # TBD
    },
    'adult': {
        'path': '../data/synthetic/adult.csv',
        'target_col': 'Education-Num',
        'epsilon': 1.0,
        'sensitivity': None,  # TBD
        'min_val': None,      # TBD
        'max_val': None       # TBD
    }
}

# ==========================================
# 3. MASTER PIPELINE LOOP
# ==========================================
print("Ξεκινάει το DiffPriv-Gateway Pipeline...\n")

processed_datasets = {}

for name, config in datasets_config.items():
    print(f"--- Επεξεργασία Dataset: [{name.upper()}] ---")
    
    # Έλεγχος αν υπάρχει το αρχείο
    if not os.path.exists(config['path']):
        print(f"Το αρχείο {config['path']} δεν βρέθηκε. Προχωράμε στο επόμενο.\n")
        continue
        
    # Έλεγχος αν η Αθηνά έχει βρει το Sensitivity
    if config['sensitivity'] is None:
        print(f"Αναμονή για Data Profiling (Λείπει το Sensitivity). Παραλείπεται...\n")
        continue
        
    # Αν όλα είναι έτοιμα, φορτώνουμε και προστατεύουμε
    try:
        df = pd.read_csv(config['path'])
        df_protected = apply_laplace_mechanism(
            df, 
            config['target_col'], 
            config['epsilon'], 
            config['sensitivity'], 
            config['min_val'], 
            config['max_val']
        )
        processed_datasets[name] = df_protected
        print(f"Επιτυχής εφαρμογή Differential Privacy στη στήλη '{config['target_col']}'!")
        print(f"Preview ({name}):\n", df_protected[[config['target_col'], f"DP_{config['target_col']}"]].head(3), "\n")
    except Exception as e:
        print(f"Σφάλμα κατά την επεξεργασία του {name}: {e}\n")

print("Το Pipeline ολοκληρώθηκε!")

Ξεκινάει το DiffPriv-Gateway Pipeline...

--- Επεξεργασία Dataset: [FIRE] ---
Επιτυχής εφαρμογή Differential Privacy στη στήλη 'Final Priority'!
Preview (fire):
    Final Priority  DP_Final Priority
0               0                0.0
1               0                0.0
2               0                2.0 

--- Επεξεργασία Dataset: [SALARY] ---
Αναμονή για Data Profiling (Λείπει το Sensitivity). Παραλείπεται...

--- Επεξεργασία Dataset: [LOANS] ---
Αναμονή για Data Profiling (Λείπει το Sensitivity). Παραλείπεται...

--- Επεξεργασία Dataset: [ADULT] ---
Αναμονή για Data Profiling (Λείπει το Sensitivity). Παραλείπεται...

Το Pipeline ολοκληρώθηκε!
